# Week 6 — Machine Learning Fundamentals and Data Preparation

**ESC 113: Computer Aided Analysis for Engineering**

This notebook is designed for a **3-hour class** and serves as the transition from Python programming to machine learning.

By the end of class, you should be able to explain the supervised-learning workflow, inspect a real engineering dataset, identify features and targets, clean and prepare data correctly, split data into training and testing sets, recognize data leakage, apply basic preprocessing, and build a simple baseline model.

## 3-hour class plan

| Time | Topic |
|---|---|
| 0:00–0:20 | What machine learning is and where engineers use it |
| 0:20–0:45 | Features, targets, samples, regression, classification |
| 0:45–1:15 | pandas review and engineering dataset inspection |
| 1:15–1:30 | Data quality: missing values, duplicates, ranges, units |
| 1:30–1:40 | Break |
| 1:40–2:05 | Train/test split, random state, and reproducibility |
| 2:05–2:25 | Scaling, preprocessing, and data leakage |
| 2:25–2:45 | Simple baseline model and prediction workflow |
| 2:45–3:00 | Engineering exercises + quiz |

### Open dataset
We use the **UCI Energy Efficiency dataset**. It contains 768 simulated building configurations, 8 design features, and two continuous outputs: Heating Load and Cooling Load.

## 1. What is machine learning?

Traditional engineering computation often starts with a known physical equation:

**inputs → equation/model → output**

Example:

P = VI

Machine learning is different. We provide examples of inputs and known outputs, and an algorithm learns a relationship from the data:

**training inputs + known outputs → learned model**

The learned model can then estimate the output for new inputs.

Machine learning does **not** replace engineering knowledge. Engineers still need to:
- define the correct problem,
- understand the units and variables,
- check whether the data are trustworthy,
- select meaningful evaluation metrics,
- recognize when the model is being used outside its valid range,
- interpret the result physically.

## 2. Common engineering machine-learning problems

### Regression
The target is a **continuous numerical quantity**.

Examples:
- concrete compressive strength in MPa,
- building heating load,
- power-plant output in MW,
- battery capacity,
- bridge deflection,
- energy consumption.

### Classification
The target is a **category**.

Examples:
- machine: healthy / faulty,
- weld: acceptable / reject,
- steel plate: fault type A / B / C,
- structural state: safe / warning / critical.

### Unsupervised learning
There may be no labeled target.

Examples:
- clustering operating conditions,
- grouping sensor behavior,
- anomaly detection.

In Weeks 6–10, we focus mainly on **supervised learning**.

## 3. Essential vocabulary

- **Sample / observation / row:** one building, experiment, machine state, or measurement.
- **Feature / predictor / input:** a variable used to make a prediction.
- **Target / response / output:** the quantity the model should predict.
- **Model:** the learned mathematical relationship.
- **Training data:** examples used to fit the model.
- **Test data:** unseen examples used for final evaluation.
- **Prediction:** a model-generated estimate for a new sample.
- **Generalization:** how well a model works on unseen data.
- **Overfitting:** learning the training data too specifically and performing poorly on new data.

### Quick concept check

For each problem, decide whether it is **regression** or **classification**.

1. Predict concrete compressive strength.
2. Decide whether a bearing is faulty.
3. Predict hourly electrical demand.
4. Classify a steel surface defect.
5. Predict battery remaining capacity.

Write your answers before running any model.

## 4. Install and import the required libraries

If you are using Google Colab, run the installation cell once.

In [ ]:
!pip -q install ucimlrepo

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", 20)

## 5. Load the UCI Energy Efficiency dataset

The dataset was created from simulations of building designs. The inputs describe building geometry and glazing, while the outputs describe heating and cooling requirements.

In [ ]:
energy = fetch_ucirepo(id=242)

X_raw = energy.data.features.copy()
y_raw = energy.data.targets.copy()

print("Feature matrix shape:", X_raw.shape)
print("Target matrix shape:", y_raw.shape)

display(X_raw.head())
display(y_raw.head())

### Dataset dimensions

If X has shape **(768, 8)**:
- 768 = number of building configurations,
- 8 = number of input features.

If y has shape **(768, 2)**:
- each building has two target values,
- Heating Load,
- Cooling Load.

## 6. Rename variables so the engineering meaning is clear

Short names such as X1 and X2 may be convenient for a paper, but descriptive names make code easier to understand and audit.

In [ ]:
feature_names = [
    "Relative_Compactness",
    "Surface_Area",
    "Wall_Area",
    "Roof_Area",
    "Overall_Height",
    "Orientation",
    "Glazing_Area",
    "Glazing_Area_Distribution"
]

target_names = ["Heating_Load", "Cooling_Load"]

X = X_raw.copy()
X.columns = feature_names

targets = y_raw.copy()
targets.columns = target_names

display(X.head())
display(targets.head())

## 7. Understand every feature before modeling

The 8 features are:

1. **Relative Compactness** — how compact the building form is.
2. **Surface Area** — exterior surface-area descriptor.
3. **Wall Area** — wall-area descriptor.
4. **Roof Area** — roof-area descriptor.
5. **Overall Height** — building-height descriptor.
6. **Orientation** — coded orientation category.
7. **Glazing Area** — amount of glazing.
8. **Glazing Area Distribution** — coded distribution of glazing.

Targets:
- **Heating Load**
- **Cooling Load**

A machine-learning model can produce numbers even when the user does not understand the variables. That does **not** make the result meaningful. Engineering interpretation must come first.

## 8. Inspect a DataFrame systematically

A useful first inspection includes:
- first rows,
- dimensions,
- column names,
- data types,
- missing values,
- duplicate rows,
- descriptive statistics.

In [ ]:
print("Rows and columns:", X.shape)
print("\nColumn names:")
print(X.columns.tolist())

print("\nData types:")
print(X.dtypes)

print("\nFirst five rows:")
display(X.head())

In [ ]:
print("Feature missing values:")
display(X.isna().sum())

print("Target missing values:")
display(targets.isna().sum())

print("Duplicated feature rows:", X.duplicated().sum())

### Important engineering question

If duplicate feature rows exist, do **not** automatically delete them.

Two rows can have identical design variables but still be legitimate repeated observations. Always understand why duplicates occur before removing them.

## 9. Descriptive statistics

Descriptive statistics help detect unreasonable ranges, unusual scales, and possible coding errors.

In [ ]:
display(X.describe().T)

In [ ]:
display(targets.describe().T)

### Guided interpretation

For each feature, look at:
- minimum,
- maximum,
- mean,
- standard deviation.

Ask:
- Does the range make sense?
- Are some variables discrete even though they are stored numerically?
- Are some features on very different numerical scales?
- Are there suspicious values such as negative areas?

## 10. Select a target

For Week 6, we will use **Heating Load** as the target.

Machine-learning notation convention:
- **X** = matrix of input features
- **y** = target vector

In [ ]:
y = targets["Heating_Load"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nFirst five target values:")
display(y.head())

## 11. Basic exploratory data analysis

Before machine learning, visualize the target and important input relationships.

In [ ]:
plt.figure(figsize=(7,4))
plt.hist(y, bins=20, edgecolor="black")
plt.xlabel("Heating Load")
plt.ylabel("Number of Buildings")
plt.title("Distribution of Heating Load")
plt.show()

### Engineering interpretation questions

1. Is the target approximately symmetric?
2. Does it appear to have one cluster or multiple groups?
3. Are there extremely large or small values?
4. Why might building designs naturally form groups?

In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(X["Relative_Compactness"], y, alpha=0.7)
plt.xlabel("Relative Compactness")
plt.ylabel("Heating Load")
plt.title("Heating Load vs Relative Compactness")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(X["Surface_Area"], y, alpha=0.7)
plt.xlabel("Surface Area")
plt.ylabel("Heating Load")
plt.title("Heating Load vs Surface Area")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(X["Glazing_Area"], y, alpha=0.7)
plt.xlabel("Glazing Area")
plt.ylabel("Heating Load")
plt.title("Heating Load vs Glazing Area")
plt.show()

## 12. Correlation as an exploratory tool

Correlation measures linear association.

Important:
- correlation does not prove causation,
- a low correlation does not mean a feature is useless,
- nonlinear relationships can have weak Pearson correlation,
- highly correlated predictors can complicate interpretation.

In [ ]:
analysis_df = X.copy()
analysis_df["Heating_Load"] = y

corr_with_target = (
    analysis_df.corr(numeric_only=True)["Heating_Load"]
    .sort_values(ascending=False)
)

display(corr_with_target)

In [ ]:
corr_matrix = analysis_df.corr(numeric_only=True)

plt.figure(figsize=(9,7))
plt.imshow(corr_matrix, aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=90)
plt.yticks(range(len(corr_matrix.index)), corr_matrix.index)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

## 13. Data-quality checks: a reusable engineering pattern

A simple reusable function helps inspect any tabular dataset.

In [ ]:
def data_quality_report(df):
    report = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "unique_values": df.nunique(),
        "min": df.min(numeric_only=False),
        "max": df.max(numeric_only=False)
    })
    return report

display(data_quality_report(X))

### Why this matters

A model may fail or mislead because of:
- missing values,
- impossible measurements,
- unit mistakes,
- duplicated records,
- categorical variables treated incorrectly,
- target leakage,
- inconsistent sensor calibration.

Machine learning begins with **data engineering**, not with calling `model.fit()`.

## 14. Demonstration: handling missing values

The UCI Energy Efficiency dataset has no missing values, so we create a small copy with missing entries only to practice preprocessing.

In [ ]:
demo = X.head(8).copy()

demo.loc[2, "Wall_Area"] = np.nan
demo.loc[5, "Glazing_Area"] = np.nan

display(demo)
print("\nMissing values:")
display(demo.isna().sum())

### Option A: remove incomplete rows

This can be reasonable when:
- only a very small fraction of rows are missing,
- missingness is not systematic,
- enough data remain.

It can be harmful when too much data are discarded.

In [ ]:
demo_drop = demo.dropna()
print("Before:", len(demo))
print("After dropna:", len(demo_drop))

### Option B: fill missing values

For numerical variables, median imputation is often a simple baseline.

In [ ]:
demo_fill = demo.copy()

for col in demo_fill.columns:
    if demo_fill[col].isna().any():
        demo_fill[col] = demo_fill[col].fillna(demo_fill[col].median())

display(demo_fill)

## 15. Train/test split

We must evaluate a model on data that were **not used to fit the model**.

A common introductory split is:
- 80% training,
- 20% testing.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("Training fraction:", len(X_train) / len(X))
print("Testing fraction:", len(X_test) / len(X))

### Why `random_state=42`?

The split is random. A fixed random state makes the exact split reproducible.

This is important because:
- students can reproduce results,
- instructors can compare outputs,
- debugging becomes easier,
- experiments can be repeated fairly.

In [ ]:
X_train_1, X_test_1, _, _ = train_test_split(
    X, y, test_size=0.20, random_state=1
)

X_train_2, X_test_2, _, _ = train_test_split(
    X, y, test_size=0.20, random_state=99
)

print("First test indices with random_state=1:")
print(X_test_1.index[:10].tolist())

print("\nFirst test indices with random_state=99:")
print(X_test_2.index[:10].tolist())

## 16. Why not train and evaluate on the same data?

If a model is evaluated on the same observations it learned from, the score can be overly optimistic.

Engineering analogy:
Testing a controller only on the exact operating conditions used to tune it does not demonstrate that it will work under new conditions.

## 17. Data leakage

**Data leakage** means information that should be unavailable during model training leaks into the training process.

Common examples:

### Leakage example 1
Scaling the full dataset before train/test splitting.

### Leakage example 2
Filling missing values using the median of the full dataset before splitting.

### Leakage example 3
Using a variable that directly contains information about the target.

### Leakage example 4
Repeatedly checking the final test set while choosing models.

Correct principle:

**Split first → learn preprocessing from training data → apply the learned preprocessing to test data.**

## 18. Feature scaling

Scaling changes numerical magnitude but does not change the ordering of the values.

Standardization:

z = (x − mean) / standard deviation

Some models benefit strongly from scaling:
- K-nearest neighbors,
- support vector machines,
- regularized linear models,
- neural networks.

Tree-based models usually do not require scaling.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training scaled mean (approximately 0):")
print(np.round(X_train_scaled.mean(axis=0), 3))

print("\nTraining scaled standard deviation (approximately 1):")
print(np.round(X_train_scaled.std(axis=0), 3))

### The important distinction

Correct:
```python
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)
```

Incorrect:
```python
scaler.fit(X)
# test-set information has influenced preprocessing
```

Why? Because the mean and standard deviation should be learned only from the training data.

## 19. Baseline models

Before building a sophisticated machine-learning model, create a simple baseline.

A baseline answers:

**Is my model actually learning something useful?**

For regression, a simple baseline can predict the mean target value for every sample.

In [ ]:
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)

baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
baseline_r2 = r2_score(y_test, baseline_pred)

print(f"Baseline MAE:  {baseline_mae:.3f}")
print(f"Baseline RMSE: {baseline_rmse:.3f}")
print(f"Baseline R²:   {baseline_r2:.3f}")

### Interpreting the baseline

A mean-prediction baseline usually has R² near 0 on unseen data.

A useful trained model should generally perform better than this baseline.

Do not judge a model by a number alone. Always compare it with:
- a baseline,
- alternative models,
- engineering requirements.

## 20. Preview: complete ML workflow with Linear Regression

Formal regression begins in Week 7. Here we fit one simple model only to see the entire workflow from start to finish.

In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

linear_pred = linear_model.predict(X_test)

linear_mae = mean_absolute_error(y_test, linear_pred)
linear_rmse = np.sqrt(mean_squared_error(y_test, linear_pred))
linear_r2 = r2_score(y_test, linear_pred)

print(f"Linear Regression MAE:  {linear_mae:.3f}")
print(f"Linear Regression RMSE: {linear_rmse:.3f}")
print(f"Linear Regression R²:   {linear_r2:.3f}")

## 21. Compare baseline and trained model

In [ ]:
comparison = pd.DataFrame({
    "Model": ["Mean Baseline", "Linear Regression"],
    "MAE": [baseline_mae, linear_mae],
    "RMSE": [baseline_rmse, linear_rmse],
    "R2": [baseline_r2, linear_r2]
})

display(comparison)

### Interpretation questions

1. Which model has the lower MAE?
2. Which has the lower RMSE?
3. Which has the higher R²?
4. Does the trained model improve over the baseline?
5. Are the errors small enough for engineering use? What additional information would you need to answer that responsibly?

## 22. Actual vs predicted plot — preview

A perfect model would place every point on the 45° line.

In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(y_test, linear_pred, alpha=0.7)

low = min(y_test.min(), linear_pred.min())
high = max(y_test.max(), linear_pred.max())
plt.plot([low, high], [low, high], "--")

plt.xlabel("Actual Heating Load")
plt.ylabel("Predicted Heating Load")
plt.title("Week 6 Preview: Actual vs Predicted")
plt.show()

## 23. Engineering error inspection

Average metrics can hide individual large errors. Engineers should inspect the worst predictions.

In [ ]:
error_table = X_test.copy()
error_table["Actual"] = y_test
error_table["Predicted"] = linear_pred
error_table["Absolute_Error"] = np.abs(
    error_table["Actual"] - error_table["Predicted"]
)

display(
    error_table
    .sort_values("Absolute_Error", ascending=False)
    .head(10)
)

### Discussion

Look at the largest-error samples.

Ask:
- Do they share similar feature values?
- Are they near the edge of the data range?
- Does the simple linear model appear inadequate for some building configurations?
- Would a nonlinear model be reasonable to try next?

## 24. Guided class exercise A — inspect the second target

Repeat the inspection for **Cooling Load**.

Tasks:
1. Create `y_cooling`.
2. Calculate min, max, mean, and standard deviation.
3. Plot a histogram.
4. Plot Cooling Load vs Relative Compactness.
5. Compare its distribution with Heating Load.

In [ ]:
# Starter code
y_cooling = targets["Cooling_Load"]

print("Mean:", y_cooling.mean())
print("Min:", y_cooling.min())
print("Max:", y_cooling.max())

## 25. Guided class exercise B — engineering feature investigation

Choose **two** features.

For each feature:
1. print its unique values or descriptive statistics,
2. create a scatter plot against Heating Load,
3. calculate its correlation with Heating Load,
4. write 2–3 sentences interpreting the relationship.

Suggested features:
- Overall Height,
- Glazing Area,
- Roof Area,
- Relative Compactness.

## 26. Guided class exercise C — reproducibility experiment

Run the same Linear Regression workflow with:

- random_state = 1
- random_state = 10
- random_state = 100

Record the test MAE for each.

Question:

Why do the results change even though the dataset and algorithm are the same?

In [ ]:
for seed in [1, 10, 100]:
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.20, random_state=seed
    )

    model = LinearRegression()
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)

    mae = mean_absolute_error(y_te, pred)
    print(f"random_state={seed:3d}  MAE={mae:.3f}")

## 27. Independent engineering exercises

Complete these during class or for practice.

### Exercise 1 — Data inspection
Write code that prints:
- number of samples,
- number of features,
- target names,
- all feature data types.

### Exercise 2 — Range checks
For every feature, calculate:
- minimum,
- maximum,
- range = max − min.

### Exercise 3 — Target comparison
Calculate the correlation between Heating Load and Cooling Load.

### Exercise 4 — Filtering
Find all buildings with:
- Overall Height greater than its median,
- Glazing Area greater than 0.

Compare their average Heating Load with the full dataset.

### Exercise 5 — Train/test ratio
Repeat the split using 70/30 instead of 80/20. Explain one advantage and one disadvantage.

### Exercise 6 — Baseline thinking
Explain why predicting the mean Heating Load is useful even though it is a poor engineering model.

### Exercise 7 — Leakage
Describe one example of leakage that could occur in a sensor-based predictive-maintenance project.

### Exercise 8 — New engineering dataset
Choose one open engineering dataset and identify:
- sample,
- features,
- target,
- regression or classification,
- possible engineering use.

## 28. Mini case study — designing a building-energy prediction system

Imagine an engineering company wants to estimate heating load early in building design.

Answer:

1. What information would the model receive?
2. What output would it generate?
3. Which stakeholders might use the prediction?
4. What errors would be costly?
5. Does this dataset represent weather, occupancy, HVAC control, and construction quality?
6. What risks arise if the model is used for a building unlike those in the training data?
7. What additional validation would be required before professional use?

This is the difference between simply **running ML code** and doing **engineering machine learning**.

## 29. Week 6 coding challenge

Create one function:

```python
def evaluate_regression(y_true, y_pred):
    ...
```

It should return a dictionary containing:
- MAE,
- RMSE,
- R².

Then use the function to evaluate both the baseline and Linear Regression models.

In [ ]:
def evaluate_regression(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred)
    }

print("Baseline:", evaluate_regression(y_test, baseline_pred))
print("Linear:", evaluate_regression(y_test, linear_pred))

## 30. Short theoretical quiz

Answer in your own words.

1. What is machine learning?
2. What is a sample?
3. What is a feature?
4. What is a target?
5. What is the difference between regression and classification?
6. Why do we separate training and testing data?
7. What is generalization?
8. What is overfitting?
9. What is data leakage?
10. Why should preprocessing parameters be learned from training data only?
11. What does `random_state` help us achieve?
12. Why is a baseline model important?
13. Name two algorithms that may benefit from feature scaling.
14. Why do tree-based models generally not require standardization?
15. Why must an engineer understand units and physical meaning before modeling?

## 31. Coding quiz

Write Python code for each task.

1. Load the UCI Energy Efficiency dataset.
2. Print the first 10 feature rows.
3. Print the number of missing values in each column.
4. Select Heating Load as `y`.
5. Create an 80/20 train/test split with `random_state=42`.
6. Print the shapes of `X_train` and `X_test`.
7. Standardize the data correctly without fitting the scaler on the test set.
8. Train a mean `DummyRegressor`.
9. Calculate its MAE.
10. Train a Linear Regression model and calculate its R².

## 32. Exit ticket

Before leaving class, write three sentences:

1. One machine-learning concept you understand well.
2. One concept that is still unclear.
3. One engineering problem you think could be addressed with supervised machine learning.

### Next week
**Week 7: Regression for Engineering Prediction**

We will study Linear Regression more carefully and evaluate predictions using MAE, RMSE, R², residuals, and actual-vs-predicted plots.

## Dataset reference

Tsanas, A. & Xifara, A. (2012). **Energy Efficiency** [Dataset]. UCI Machine Learning Repository. DOI: 10.24432/C51307.

UCI dataset page:
https://archive.ics.uci.edu/dataset/242/energy+efficiency